**1. Install the libraries required to import the model from Hugging face**

In [1]:
!pip install torch transformers datasets peft accelerate rouge-score evaluate pandas

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=278756eabc6de48fe4e9a25999fb5a4f02473ea16dd2c8dc8f920390eb9ffe03
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


**2. Read the data after the data is cleaned and stored in a csv file**

In [2]:
import pandas as pd
df = pd.read_csv('/content/sample_data/output_20260415064406.csv')
df.head()

,id,findings,labels
0,3997,"Heart size within normal limits. Small, nodula...","No acute findings, no evidence for active TB."
1,3996,The lungs are clear. Heart size is normal. No ...,Clear lungs. No acute cardiopulmonary abnormal...
2,3995,The cardiomediastinal silhouette and pulmonary...,1. Interval resolution of bibasilar airspace d...
3,3994,Similar mild cardiomegaly. Of the pulmonary va...,Mild cardiomegaly with XXXX of early failure.
4,3993,The heart is mildly enlarged. Left hemidiaphra...,Borderline cardiomegaly without acute disease.


**3. Include additional text in the findings column which will summarize the text.**

In [3]:
df["findings"] = "summarize medical report and return summary in no more than 2 sentences: " + df["findings"]

In [4]:
df.head()
print(df['findings'][0])

summarize medical report and return summary in no more than 2 sentences: Heart size within normal limits. Small, nodular opacity in the right upper lobe. This does not look like an acute infiltrate, and more XXXX represents a granuloma. No pneumothorax or effusions.


**4. Download the model from huggingface**

In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

**5. Use Lora to train the model**

In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q", "v"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)

**6. Set the training arguments**

In [7]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./medical_model",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch"
)

In [8]:
df.columns

Index(['id', 'findings', 'labels'], dtype='object')

**7. Split the data into train and test**

In [9]:
from transformers import Trainer
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Convert pandas DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split the dataset into train and test sets using the Dataset's own split method
split_datasets = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_datasets["train"]
test_dataset = split_datasets["test"]



**8. Tokenize the data**

In [12]:
# Define the tokenization function
def tokenize_function(examples):
  # return tokenizer(examples["findings"],
  #                  text_target=examples["labels"],
  #                  truncation=True)
  return tokenizer(
        [str(f) for f in examples["findings"]],
        text_target=[str(l) for l in examples["labels"]],
        truncation=True
    )

# Apply tokenization to the datasets
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Remove the original text columns as they are no longer needed
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["id", "findings"])
tokenized_test_dataset = tokenized_test_dataset.remove_columns(["id", "findings"])

# Make sure the 'labels' column is correctly set for training
tokenized_train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset
)

trainer.train()

Map:   0%|          | 0/1525 [00:00<?, ? examples/s]

Map:   0%|          | 0/170 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,8.344776,8.025359
2,8.231919,7.823103
3,8.185046,7.780329


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=4575, training_loss=8.391514672451333, metrics={'train_runtime': 2125.101, 'train_samples_per_second': 2.153, 'train_steps_per_second': 2.153, 'total_flos': 124407333586944.0, 'train_loss': 8.391514672451333, 'epoch': 3.0})

**9. Save the model**

In [13]:
import os

output_dir = "/content/medical_model/final-artifacts-g5/medical_summarizer_model_g5"
os.makedirs(output_dir, exist_ok=True)

merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/medical_model/final-artifacts-g5/medical_summarizer_model_g5/tokenizer_config.json',
 '/content/medical_model/final-artifacts-g5/medical_summarizer_model_g5/tokenizer.json')

**10. Run predictions**

In [14]:
import pandas as pd
import torch
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# load model
model_path = "/content/medical_model/final-artifacts-g5/medical_summarizer_model_g5"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

device = torch.device("cpu")
model.to(device)

# load data
test_df = pd.read_csv("/content/sample_data/output_validation_20260415071239.csv")

inputs = test_df["findings"].tolist()
references = test_df["labels"].tolist()

# predict
predictions = []

for text in inputs:

    text = "summarize medical report and return summary in no more than 2 sentences: " + text

    tokens = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    output = model.generate(
        **tokens,
        max_length=64,
        num_beams=4
    )

    pred = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    predictions.append(pred)

print(predictions)
# rouge
rouge = evaluate.load("rouge")

results = rouge.compute(
    predictions=predictions,
    references=references
)

print(results)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


['No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary cardiopulmonary abnormality.', 'No acute cardiopulmonary abnormality.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary abnormality.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute

{'rouge1': np.float64(0.47109690237365653), 'rouge2': np.float64(0.3210758829601464), 'rougeL': np.float64(0.47031940910395825), 'rougeLsum': np.float64(0.47072452458306135)}


In [17]:
import zipfile
import os
from google.colab import files

folder_to_compress = "/content/medical_model/final-artifacts-g5/medical_summarizer_model_g5"
output_zip_name = "medical_summarizer_model_g5.zip"

# Create a ZipFile object
with zipfile.ZipFile(output_zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_in_folder in os.walk(folder_to_compress):
        for file in files_in_folder:
            file_path = os.path.join(root, file)
            # Add file to zip, preserving directory structure relative to folder_to_compress
            zipf.write(file_path, os.path.relpath(file_path, folder_to_compress))

print(f"'{folder_to_compress}' successfully compressed to '{output_zip_name}'")

# Download the zip file
files.download(output_zip_name)

'/content/medical_model/final-artifacts-g5/medical_summarizer_model_g5' successfully compressed to 'medical_summarizer_model_g5.zip'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>